<a href="https://colab.research.google.com/github/psh-computer/Machine-Learning/blob/main/day02_01_text_data_tokenizer_v1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week01 - 혼합 코퍼스 토크나이저 만들기 (Track A) + 사전학습 토크나이저 비교 (Track B)
## 수업목표(week01)

1. "토크나이저가 모델의 입력 규격(Interface)"이라는 사실을 코드로 체득한다.
2. (Track A) 공개 데이터셋을 섞어 BPE 토크나이저를 직접 학습하고, 특수 토큰 규격을 고정한다.
3. (Track B) GPT-2 토크나이저를 로드해, 같은 문장이라도 토큰 분해가 달라짐을 관찰한다.

# 왜 Day02가 중요할까?
LLM 실습에서 "데이터가 좋아도 모델이 안 배운다"는 문제가 자주 발생합니다.

그런데 그 원인이 모델 구조가 아니라 __토크나이저 규격 불일치(특수 토큰, vocab size, padding 처리)__ 인 경우가 많습니다.

- 특수 토큰을 추가했는데 임베딩 크기를 안 늘리면 --> CUDA device-side assert 같은 에러로 터짐
- `<bos>`/`[BOS]`를 섞어 쓰면 --> 모델은 "시작 토큰"을 제대로 학습하지 못함
- 데이터가 너무 단일하면 --> 토큰 분포가 편향되어 이후 단계(멀티모달/다운스트림)에 악영향

Day02에서는 이 문제를 __실제로 겪지 않도록__, 특수 토큰 규격을 명학히 정하고 끝까지 유지합니다.

In [ ]:
# ---------------------------------------------------------------
# [환경 설정] 패키지 설치 및 버전 고정
# 버전을 명시해야 다른 환경에서도 동일하게 재현 가능하다.
# numpy<2 제약을 두지 않는 이유 : Colab은 이미 numpy >= 2 기반 패키지들이
# 메모리에 로드돼 있어서, 다운그레이드 시 바이너리 충돌이 발생한다.
# ---------------------------------------------------------------

!pip -q install \
    "datasets==3.6.0" \
    "transformers==4.51.3" \
    "tokenizers==0.21.1"

# 실제 설치된 버전을 출력해 두면 나중에 재현 문제가 생겼을 때 비교할 수 있다.
import numpy as np, datasets, transformers, tokenizers
print("numpy        =", np.__version__)
print("datasets     =", datasets.__version__)
print("transformers =", transformers.__version__)
print("tokenzizers  =", tokenizers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
numpy        = 2.1.3
datasets     = 3.6.0
transformers = 4.51.3
tokenzizers  = 0.23.1


## 1) 사용할 데이터(스트리밍 샘플링)

__핵심 포인트__:Colab은 디스크/시간이 제한적이므로, "전체 다운로드" 대신 `streaming=True`로 **필요한 만큼만 샘플링**합니다.

이번 과정에서 사용할 텍스트 분포는 다음 3종입니다.

1. __동화/짧은 이야기__ : 작은 모델이 문장 구조를 빨리 배우는 데 유리
2. **교육/설명 문장** : "정의/설명" 스타일 텍스트 분포를 섞어 범용성을 증가
3. **이미지 캡션(Week05 대비)** : 멀티모달(캡셔닝) 단계에서 언어 분포가 너무 달라지는 문제를 완화

In [ ]:
import os, random
from itertools import islice
from datasets import load_dataset
import time

SEED = 42
random.seed(SEED) # 셔플 재현성을 위해 시드 고정

def stream_text_samples(dataset_id: str, split: str, text_key: str, n: int):
  """
  HuggingFace 스트리밍 데이터셋에서 text_key 칼럼을 n개까지 뽑는다.
  streaming=True --> 전체를 다운로드하지 않고 필요한 만큼만 가져온다.
  """

  ds = load_dataset(dataset_id, split=split, streaming=True)
  out = []
  for ex in islice(ds, n):            # 최대 n개까지만 순회
    t = ex.get(text_key, None)        # 컬럼이 없으면 None 반환 (KetError) 방지
    if isinstance(t, str) and len(t) > 0: # 빈 문자열 None 제외
      out.append(t)
  return out

# ---- 데이터 수집 ----
stt = time.time()

# 50_000 : Python 숫자 리터럴 구분자 --> 50000과 동일, 가독성을 위해 사용
tinystories = stream_text_samples("roneneldan/TinyStories", "train", "text", n=50_000)
fineweb_edu = stream_text_samples("HuggingFaceFW/fineweb-edu", "train", "text", n=30_000)

# Flickr8k : HF Hub에 올린 버전마다 컬럼명이 'caption' / 'captions' 등으로 다를 수 있다.
# 로드 실패 시 캡션 없이 진행하도록 try/except로 감싼다.

flickr_caps = []
try:
  flickr8k = "ariG23498/flickr8k"
  flickr_stream = load_dataset(flickr8k, split='train', streaming=True)
  first = next(iter(flickr_stream)) # 첫 샘플로 컬럼명을 확인

  # 컬럼명 자동 탐지 : 'caption 우선 없으면 'captions' 시도
  cap_key = "caption" if "caption" in first else ("captions" if "captions" in first else None)
  assert cap_key is not None, f"caption 컬럼을 찾지 못했습니다. colums = {list(first.keys())}"

  for ex in islice(load_dataset(flickr8k, split='train', streaming = True), 20_000):
    c = ex.get(cap_key, None)
    if isinstance(c, list):
      flickr_caps.extend([x for x in c if isinstance(x, str) and len(x) > 0])
    elif isinstance(c, str) and len(c) > 0:
      flickr_caps.append(c)

  flickr_caps = flickr_caps[:20_000] # 최대 2만개로 제한

except Exception as e:
  print("[주의] Flickr8k 로드 실패 -> 캡션 없이 진행합니다.")
  print('원인:' , e)

# ----수집 데이터 확인----
print("TinyStories :", len(tinystories))
print('FineWeb-edu :', len(fineweb_edu))
print("Flickr8k    :", len(flickr_caps))
print('process-time:', time.time()-stt)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/710 [00:00<?, ?B/s]

TinyStories : 49988
FineWeb-edu : 30000
Flickr8k    : 20000
process-time: 150.3396737575531


## 2) Track A: BPE 토크나이저 학습

특수 토큰 규격(고정)

- `<pad>`:배치 padding
- `<bos>`:문장 시작
- `<eos>`:문장 끝
- `<unk>`:OOV(사전에 없는 토큰)

**중요** : Week02에서 from-scratch LLM을 만들 때

`vocab_size`는 "토크나이저 vocab 크기"와 **정확히 일치**해야 합니다.

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from tokenizers.processors import TemplateProcessing

stt = time.time()

# 특수 토큰 목록 - 순서가 곧 ID가 된다: <pad>=0, <bos>=1, <eos>=2, <unk>=3
SPECIALS = ['<pad>', '<bos>', '<eos>', '<unk>']
VOCAB_SIZE = 51_000 # 실습용으로 충분한 크기: GPT-2는 50257

# 세 코퍼스를 합쳐 셔플 -> 특정 도메인 편향 없이 BPE 학습
mix_texts = tinystories + fineweb_edu + flickr_caps
random.shuffle(mix_texts)

# unk_token을 모델에 전달해야 OOV(사전 외 문자) 처리가 된다.
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

# NFKC 정규화: 'a"(전각) -> 'a'(반각) 같은 유니코드 변형을 통일
tokenizer.normalizer = normalizers.Sequence([normalizers.NFKC()])

# ByteLevel: 공백을 버리지 않고 다음 단어 앞에 G(U+0120)로 붙여 보존
# --> 디코딩 시 G를 공백으로 복원해 원문 재현 가능
# add_prefix_space == True: 문장 첫 단어도 G를 붙여 일관성 유지
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIALS,  # 특수 토큰은 병합 대상에서 제외, 낮은 ID 보장
    min_frequency=2           # 2회 미만 등장한 쌍은 병합하지 않음 (노이즈 제거)
)

def batch_iter(texts, bs=1000):
  """대용량 텍스트를 bs 단위 배치로 쪼개 메모리를 아끼며 학습에 넘긴다."""
  for i in range(0, len(texts), bs):
    yield texts[i : i + bs]

# 배치 이터레이터로 BPS 학습 - 전체 리스트를 한 번에 넘기면 OOM 위험
tokenizer.train_from_iterator(batch_iter(mix_texts), trainer = trainer)

# TemplateProcessing: encode() 호출 시마다 <bos> . . . <eos>를 자동으로 감싼다.
# $A = 입력 시퀀스, $B:1 = 두 번째 시퀀스(pair 태스크용, segment id=1)
tokenizer.post_processor = TemplateProcessing(
    single='<bos> $A <eos>',
    pair='<bos> $A <eos> $B:1 <eos>:1',
    special_tokens=[
        ('<bos>', tokenizer.token_to_id("<bos>")),
        ('<eos>', tokenizer.token_to_id("<eos>")),
    ],
)

# 학습 결과 확인
print('vocab size  :', tokenizer.get_vocab_size())
print('special IDs :', {s: tokenizer.token_to_id(s) for s in SPECIALS})
print('process-time:', time.time()-stt)

vocab size  : 51000
special IDs : {'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3}
process-time: 133.92129373550415


## 3) 저장 & 빠른 검증

산출물:

- `/content/week01_artifacts/mix_bpe_tokenizer.json`

In [ ]:
OUT_DIR = "/content/day02_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)   # 디렉터리가 없으면 생성

tok_path = os.path.join(OUT_DIR, "mix_bpe_tokenizer.json")
tokenizer.save(tok_path)
print("Saved:", tok_path)

# ---인코딩 -> 디코딩 왕복 검증 ---
test_text = "A dog jumps over a log near the beach."
enc = tokenizer.encode(test_text)

print('\n(원문)')
print(test_text)

print('\n(토큰 목록)')
print(enc.tokens)       # 문자열 토큰 리스트 - <bos>/<eos> 포함 여부 확인

print("\n(토큰 ID)")
print(enc.ids)          # 정수 ID 리스트 - 모델 입력값

print('\n(디코딩 복원 - 특수 토큰 포함)')
# skip.special_tokens=False: <bos>/<eos>가 복원 문자열에 포함되는지 확인
print(tokenizer.decode(enc.ids, skip_special_tokens=False))

print('\n[디코딩 복원 - 특수 토큰 제거]')
# skip_special_tokens=True: 실제 추론 출력에서 쓰는 방식
print(tokenizer.decode(enc.ids, skip_special_tokens=True))

Saved: /content/day02_artifacts/mix_bpe_tokenizer.json

(원문)
A dog jumps over a log near the beach.

(토큰 목록)
['<bos>', 'ĠA', 'Ġdog', 'Ġjumps', 'Ġover', 'Ġa', 'Ġlog', 'Ġnear', 'Ġthe', 'Ġbeach', '.', '<eos>']

(토큰 ID)
[1, 284, 955, 8118, 631, 209, 2869, 1233, 211, 3586, 17, 2]

(디코딩 복원 - 특수 토큰 포함)
<bos> ĠA Ġdog Ġjumps Ġover Ġa Ġlog Ġnear Ġthe Ġbeach . <eos>

[디코딩 복원 - 특수 토큰 제거]
ĠA Ġdog Ġjumps Ġover Ġa Ġlog Ġnear Ġthe Ġbeach .


In [ ]:
OUT_DIR = '/content/week01_artifacts'
os.makedirs(OUT_DIR, exist_ok=True)

tok_path = os.path.join(OUT_DIR, "mix_bpe_tokenizer.json")
tokenizer.save(tok_path)
print("Saved:", tok_path)

test_text = "A dog jumps over a log near the beach."

print('\n(원문)')
print(test_text)
print('\n[토큰 ID 일부]')
print(enc.ids[:40], "...")
print('\n[디코드(복원)]')
print(tokenizer.decode(enc.ids))

Saved: /content/week01_artifacts/mix_bpe_tokenizer.json

(원문)
A dog jumps over a log near the beach.

[토큰 ID 일부]
[1, 284, 955, 8118, 631, 209, 2869, 1233, 211, 3586, 17, 2] ...

[디코드(복원)]
ĠA Ġdog Ġjumps Ġover Ġa Ġlog Ġnear Ġthe Ġbeach .
